<a href="https://colab.research.google.com/github/S-VIGHNESH18/projects/blob/project/Energy_Consumption_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# @title Install Streamlit and pyngrok if not already installed
!pip install streamlit "pyngrok>=5.0.0" -qq

In [6]:
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import requests # For OpenWeatherMap API
import os # Added for file operations
from sklearn.ensemble import RandomForestRegressor # NEW: For Random Forest
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score # For model evaluation

# The Streamlit application code
app_code = """
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import requests # For OpenWeatherMap API
import os # Added for file operations
from sklearn.ensemble import RandomForestRegressor # NEW: For Random Forest
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score # For model evaluation

# Initialize session state for login and username
if "logged_in" not in st.session_state:
    st.session_state["logged_in"] = False
if "username" not in st.session_state:
    st.session_state["username"] = None

# =========================
# LOGIN FUNCTIONALITY
# =========================
# Define user credentials
USERS = {
    "admin": "1234",
    "vighnesh": "password"
}

if not st.session_state["logged_in"]:
    st.sidebar.subheader("Login")
    username_input = st.sidebar.text_input("Username", key="login_username")
    password_input = st.sidebar.text_input("Password", type="password", key="login_password")

    if st.sidebar.button("Login", key="login_button"):
        if username_input in USERS and USERS[username_input] == password_input:
            st.session_state["logged_in"] = True
            st.session_state["username"] = username_input
            st.sidebar.success(f"Logged in as {username_input}!")
            # st.experimental_rerun() # Removed to prevent potential IndentationError
        else:
            st.sidebar.error("Incorrect username or password.")
else:
    st.sidebar.write(f"Logged in as: {st.session_state['username']}")
    if st.sidebar.button("Logout", key="logout_button"):
        st.session_state["logged_in"] = False
        st.session_state["username"] = None
        st.sidebar.info("Logged out successfully.")
        # st.experimental_rerun() # Removed to prevent potential IndentationError

# =========================
# PAGE CONFIG
# =========================
st.set_page_config(
    page_title="EcoWatt",
    page_icon="⚡",
    layout="wide"
)

st.title("⚡ EcoWatt")
st.subheader("Predict • Save • Sustain")

# Define history file globally within the app code
history_file = "prediction_history.csv"

# =========================
# WEATHER API INTEGRATION
# =========================
WEATHER_API_KEY = "40d15d9b8860af245b44d15ee812f496" # Active OpenWeatherMap API Key for weather integration
WEATHER_BASE_URL = "http://api.openweathermap.org/data/2.5/weather"

def get_weather_data(city_name):
    params = {
        "q": city_name,
        "appid": WEATHER_API_KEY,
        "units": "metric"
    }
    try:
        response = requests.get(WEATHER_BASE_URL, params=params)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        return response.json()
    except requests.exceptions.HTTPError as e:
        st.error(f"Error fetching weather: {e.response.status_code} - {e.response.text}")
        return None
    except requests.exceptions.RequestException as e:
        st.error(f"Error making weather API request: {e}")
        return None

# Add UI for weather display
st.subheader("Current Weather Information")
weather_city = st.text_input("Enter city for weather information", "Kottayam")

if st.button("Get Weather"):
    if weather_city:
        weather_data = get_weather_data(weather_city)
        if weather_data:
            st.write(f"**City:** {weather_data['name']}")
            st.write(f"**Temperature:** {weather_data['main']['temp']} °C")
            st.write(f"**Feels Like:** {weather_data['main']['feels_like']} °C")
            st.write(f"**Humidity:** {weather_data['main']['humidity']}%")
            st.write(f"**Conditions:** {weather_data['weather'][0]['description'].capitalize()}")
        else:
            st.warning(f"Could not retrieve weather data for {weather_city}.")
    else:
        st.warning("Please enter a city name.")

# =========================
# SAMPLE DATASET
# =========================
np.random.seed(42)
data_size = 200

temperature = np.random.randint(20, 40, data_size)
time_hour = np.random.randint(0, 24, data_size)
humidity = np.random.randint(40, 90, data_size)
power_rating = np.random.randint(100, 2500, data_size)
usage_duration = np.random.uniform(0.5, 10, data_size)
num_devices = np.random.randint(1, 10, data_size)
occu_pancy = np.random.randint(1, 6, data_size)

energy_consumption = (
    (power_rating * usage_duration * num_devices) / 1000
    + 0.3 * temperature
    + 0.1 * humidity
    + 0.5 * occu_pancy
    + np.random.normal(0, 2, data_size)
)

df = pd.DataFrame({
    "Temperature": temperature,
    "Time": time_hour,
    "Humidity": humidity,
    "PowerRating": power_rating,
    "UsageDuration": usage_duration,
    "NumDevices": num_devices,
    "Occupancy": occu_pancy,
    "EnergyConsumption": energy_consumption
})

# Store initial df in session state if not present
if 'original_df' not in st.session_state:
    st.session_state.original_df = df.copy()

# =========================
# TRAIN MODEL (Initial Training)
# =========================
# Function to train the model
def train_model(dataframe):
    X_train = dataframe.drop("EnergyConsumption", axis=1)
    y_train = dataframe["EnergyConsumption"]
    model_new = LinearRegression()
    model_new.fit(X_train, y_train)
    return model_new

# Initial model training
model = train_model(df)

# =========================
# SIDEBAR INPUT
# =========================
st.sidebar.header("Input Parameters")

temp = st.sidebar.slider("Temperature (°C)", 15, 45, 30)
hour = st.sidebar.slider("Hour of Usage", 0, 23, 18)
humidity = st.sidebar.slider("Humidity (%)", 30, 100, 65)
power = st.sidebar.slider("Power Rating (W)", 100, 3000, 1500)
duration = st.sidebar.slider("Usage Duration (hrs)", 1.0, 24.0, 4.0)
devices = st.sidebar.slider("Number of Devices", 1, 15, 5)
occu_pancy = st.sidebar.slider("Occupancy", 1, 10, 3)

# Add a button to trigger prediction
predict_button = st.sidebar.button("Predict Energy")

# Initialize prediction variables outside the button block to avoid NameError
predicted_energy = None
cost = None
cO2 = None
peak = None
saving = None
eco_score = None

if predict_button:
    # =========================
    # PREDICTION - Linear Regression
    # =========================
    input_df = pd.DataFrame(
        {
        "Temperature": [temp],
        "Time": [hour],
        "Humidity": [humidity],
        "PowerRating": [power],
        "UsageDuration": [duration],
        "NumDevices": [devices],
        "Occupancy": [occu_pancy]
    })

    predicted_energy = model.predict(input_df)[0]

    # COST
    tariff = 8.4
    cost = predicted_energy * tariff

    # CO2
    emission_factor = 0.83
    co2 = predicted_energy * emission_factor

    # Peak Hour
    if 18 <= hour <= 21:
        peak = "6 PM - 9 PM"
    elif 12 <= hour <= 15:
        peak = "12 PM - 3 PM"
    elif 21 < hour <= 23:
        peak = "9 PM - 11 PM"
    else:
        peak = "Low Demand"

    # Saving Potential
    baseline = predicted_energy * 1.14
    saving = ((baseline - predicted_energy) / baseline) * 100

    # Eco Score
    eco_score = max(0, 100 - (predicted_energy * 1.5 + co2 * 0.5))

    # =========================
    # SAVE PREDICTION HISTORY
    # =========================
    def save_prediction(data):
        df_new = pd.DataFrame([data])
        if os.path.exists(history_file):
            df_old = pd.read_csv(history_file)
            df_combined = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df_combined = df_new
        df_combined.to_csv(history_file, index=False)

    prediction_record = {
        "Timestamp": pd.to_datetime('now'), # Added Timestamp for better history tracking
        "Temperature_Input": temp,
        "Humidity_Input": humidity,
        "PowerRating_Input": power,
        "UsageDuration_Input": duration,
        "NumDevices_Input": devices,
        "Occupancy_Input": occu_pancy,
        "Predicted_Energy_kWh": predicted_energy,
        "Cost_INR": cost,
        "CO2_kg": co2,
        "PeakHour_Category": peak
    }

    save_prediction(prediction_record)

    # =========================
    # OUTPUT DASHBOARD
    # =========================
    col1, col2, col3 = st.columns(3)

    col1.metric("⚡ Energy Usage", f"{predicted_energy:.2f} kWh")
    col2.metric("💰 Cost", f"₹{cost:.2f}")
    col3.metric("🌍 CO₂", f"{co2:.2f} kg")

    col4, col5, col6 = st.columns(3)

    col4.metric("⏰ Peak Hour", peak)
    col5.metric("📉 Saving Potential", f"{saving:.2f}%")
    col6.metric("🌱 Eco Score", f"{eco_score:.1f}/100")

    # =========================
    # RECOMMENDATIONS
    # =========================
    st.subheader("Recommendations")

    if predicted_energy > 25:
        st.warning("High energy usage detected. Reduce appliance overlap.")

    if temp > 32:
        st.info("High temperature detected. Optimize AC usage.")

    if eco_score < 60:
        st.error("Eco score is low. Improve efficiency.")

    report = pd.DataFrame({
        "Predicted Energy":[predicted_energy],
        "Cost":[cost],
        "CO2":[co2],
        "Peak Hour":[peak],
        "Saving %":[saving],
        "Eco Score":[eco_score]
    })

    csv = report.to_csv(index=False)

    st.download_button(
        label="Download Report",
        data=csv,
        file_name="energy_report.csv",
        mime="text/csv"
    )

    if predicted_energy > 30:
        st.error("ALERT: Very high energy usage!")

    if cost > 300:
        st.warning("Electricity bill may be high today!")

    if co2 > 25:
        st.warning("Carbon footprint exceeds safe threshold.")
else: # If predict button is not clicked, display a message
    st.info("Adjust the parameters in the sidebar and click 'Predict Energy' to see the results.")


# =========================
# PREDICTION HISTORY DASHBOARD
# =========================
st.subheader("Prediction History")

if os.path.exists(history_file):
    history_df = pd.read_csv(history_file)
    st.write("Current Prediction History (editable):")

    # Use st.data_editor for interactive editing and deletion
    edited_history_df = st.data_editor(
        history_df,
        use_container_width=True,
        num_rows="dynamic", # Allows adding/deleting rows
        key="editable_prediction_history"
    )

    # Check if the edited DataFrame is different from the original and save if it is
    if not edited_history_df.equals(history_df):
        st.success("Changes detected! Saving updated history...")
        edited_history_df.to_csv(history_file, index=False)
        st.rerun() # Rerun to reflect changes and update the display

    # Optional: Plotting history
    if not history_df.empty:
        st.line_chart(history_df['Predicted_Energy_kWh'])
    else:
        st.info("History is empty.")

    # Add a button to clear all history
    if st.button("Clear All History", help="Permanently delete all prediction records."):
        if os.path.exists(history_file):
            os.remove(history_file)
            st.success("Prediction history cleared!")
            st.rerun()
else:
    st.info("No prediction history yet. Make some predictions!")


# =========================
# CHARTS
# =========================
st.subheader("Analytics")

fig, ax = plt.subplots(figsize=(8,4))
ax.plot(df["EnergyConsumption"][:30], marker='o')
ax.set_title("Energy Consumption Trend")
ax.set_xlabel("Days")
ax.set_ylabel("kWh")
st.pyplot(fig)

# Appliance Consumption Chart
appliances = ["AC", "Fan", "TV", "Fridge", "Lights"]
values = [40, 15, 10, 20, 15]

fig2, ax2 = plt.subplots(figsize=(6,4))
ax2.bar(appliances, values)
ax2.set_title("Appliance Consumption (%)")
st.pyplot(fig2)

# =========================
# Random Forest MODEL FOR FUTURE PREDICTION (Admin Only)
# =========================
if st.session_state.get("logged_in") and st.session_state.get("username") == "admin":
    st.subheader("Time Series Forecasting with Random Forest")
    st.success("Admin access granted for Random Forest features! Now you can train, predict, and evaluate.") # Verification

    # Function to create lagged features for Random Forest
    def create_lagged_features_rf(data, lag_length):
        X, y = [], []
        if data.ndim > 1:
            data = data.flatten()
        for i in range(len(data) - lag_length):
            X.append(data[i:(i + lag_length)])
            y.append(data[i + lag_length])
        return np.array(X), np.array(y)

    # Prepare data for Random Forest (use 'EnergyConsumption' directly)
    rf_data = df['EnergyConsumption'].values # 1D numpy array

    lag_length = 10 # Define lag length for Random Forest input (similar to seq_length)

    if len(rf_data) >= lag_length + 1: # Ensure enough data
        X_rf, y_rf = create_lagged_features_rf(rf_data, lag_length)

        # Define Random Forest Model
        if 'rf_model' not in st.session_state:
            rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1) # n_jobs=-1 for parallel processing
            with st.spinner('Training Random Forest model... This might take a moment.'):
                rf_model.fit(X_rf, y_rf)
            st.session_state.rf_model = rf_model
            st.session_state.lag_length_rf = lag_length
            st.success("Random Forest Model Trained and Ready!")
        else:
            rf_model = st.session_state.rf_model
            lag_length = st.session_state.lag_length_rf
            st.info("Random Forest Model Loaded from session state.")

        # Random Forest Future Prediction UI
        st.sidebar.subheader("Random Forest Future Prediction")
        n_future_steps = st.sidebar.slider("Number of Future Steps (Random Forest)", 1, 30, 7)
        predict_future_button = st.sidebar.button("Predict Future Energy")

        if predict_future_button:
            # Get the last 'lag_length' data points for prediction
            last_sequence = rf_data[-lag_length:]

            future_predictions = []
            current_prediction_input = last_sequence

            for _ in range(n_future_steps):
                # Predict the next step
                next_step_prediction = rf_model.predict(current_prediction_input.reshape(1, -1))[0] # Reshape for single sample prediction
                future_predictions.append(next_step_prediction)

                # Update the input sequence by removing the first element and adding the new prediction
                current_prediction_input = np.append(current_prediction_input[1:], next_step_prediction)

            # Create a DataFrame for plotting
            last_index = df.index[-1]
            future_indices = pd.RangeIndex(start=last_index + 1, stop=last_index + 1 + n_future_steps)

            future_df_plot = pd.DataFrame({
                'Predicted Energy Consumption (kWh)': np.array(future_predictions)
            }, index=future_indices)

            # Combine historical and future data for plotting
            plot_data = pd.concat([df['EnergyConsumption'], future_df_plot['Predicted Energy Consumption (kWh)']])
            plot_data_df = pd.DataFrame({'Energy Consumption': plot_data})

            st.subheader(f"Random Forest Predicted Energy Consumption for next {n_future_steps} steps")
            st.dataframe(future_df_plot)
            st.line_chart(plot_data_df)
    else:
        st.warning(f"Not enough data in the current dataset to train Random Forest model with lag length {lag_length}. Need at least {lag_length + 1} data points to create features.")

    # =========================
    # Random Forest MODEL EVALUATION (Admin Only)
    # =========================
    st.subheader("Random Forest Model Evaluation")

    if 'rf_model' in st.session_state and len(rf_data) >= lag_length + 1:
        # Ensure we have consistent X_eval and y_eval for evaluation
        X_eval, y_eval = create_lagged_features_rf(rf_data, lag_length)

        # Make predictions on the training/evaluation set
        rf_predictions = st.session_state.rf_model.predict(X_eval)

        mae = mean_absolute_error(y_eval, rf_predictions)
        mse = mean_squared_error(y_eval, rf_predictions)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_eval, rf_predictions)

        st.write(f"Mean Absolute Error (MAE): {mae:.2f}")
        st.write(f"Mean Squared Error (MSE): {mse:.2f}")
        st.write(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
        st.write(f"R-squared (R2 Score): {r2:.2f}")

        st.markdown("--- More detailed evaluation (e.g., on a hold-out test set) would be performed in a production setup ---")
    else:
        st.info("Random Forest model not yet trained or insufficient data for evaluation.")
else:
    if st.session_state.get("logged_in"):
        st.info("You must be logged in as 'admin' to access Random Forest Future Prediction and Evaluation.")
    else:
        st.info("Log in as 'admin' to access Random Forest Future Prediction and Evaluation.")

# Dataset Preview
st.subheader("Sample Dataset (Editable)")

# Allow users to edit the generated DataFrame
edited_df = st.data_editor(
    st.session_state.original_df,
    use_container_width=True,
    num_rows="dynamic",
    key="editable_sample_dataset"
)

# Check if the edited DataFrame is different from the original and update if it is
if not edited_df.equals(st.session_state.original_df):
    st.session_state.original_df = edited_df.copy()
    df = edited_df # Update the main df to reflect changes
    st.success("Sample dataset updated! Model will be retrained with new data.")
    model = train_model(df) # Retrain the model with the modified data
    st.rerun() # Rerun to reflect changes
else:
    df = st.session_state.original_df # Ensure df always points to the latest (potentially edited) data


###csv file upload

st.sidebar.subheader("Upload Dataset")

uploaded_file = st.sidebar.file_uploader(
    "Upload CSV",
    type=["csv"]
)

if uploaded_file is not None:
    new_df = pd.read_csv(uploaded_file)
    # Validate columns if necessary, or just proceed
    required_columns_check = [
        "Temperature", "Time", "Humidity", "PowerRating",
        "UsageDuration", "NumDevices", "Occupancy", "EnergyConsumption"
    ]
    if all(col in new_df.columns for col in required_columns_check):
        df = new_df.copy() # Update the main df
        st.session_state.original_df = df.copy() # Update the session state for editable data
        st.success("Dataset uploaded successfully! Model will be retrained with new data.")
        model = train_model(df) # Retrain model with uploaded data
        st.dataframe(df.head())
        st.rerun() # Rerun to reflect changes throughout the app
    else:
        st.error("Uploaded CSV is missing one or more required columns.")
        st.info(f"Required columns: {', '.join(required_columns_check)}")
"""

# Write the Streamlit app code to a file
with open("app.py", "w") as f:
    f.write(app_code)

# Import necessary libraries for running Streamlit with ngrok
from pyngrok import ngrok
import subprocess
import os
import socket
import time
import psutil
import requests # Import requests for direct API call

# Function to find a free port
def find_free_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(('', 0))
        return s.getsockname()[1]

# Function to kill processes running on a given port
def kill_process_on_port(port):
    for proc in psutil.process_iter(['pid', 'name', 'connections']):
        try:
            for conn in proc.connections(kind='inet'):
                # Check for both IPv4 and IPv6 addresses
                if conn.laddr.port == port and (conn.laddr.ip == '127.0.0.1' or conn.laddr.ip == '::1' or conn.laddr.ip == '0.0.0.0'):
                    print(f"Killing process {proc.pid} ({proc.name()}) on port {port}")
                    proc.terminate()
                    proc.wait(timeout=10) # Increased timeout
        except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
            pass

# --- Enhanced Cleanup before starting ---

# 1. Kill any running Streamlit processes
for proc in psutil.process_iter(['pid', 'name']):
    if 'streamlit' in proc.info['name'].lower(): # .lower() for robustness
        print(f"Killing existing streamlit process {proc.info['pid']} ({proc.info['name']})")
        try:
            proc.terminate()
            proc.wait(timeout=10) # Increased timeout
        except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
            pass

# 2. Kill any outstanding ngrok processes
print("Killing all ngrok processes...")
ngrok.kill()

# 3. Remove ngrok configuration file to clear cached state (often ~/.ngrok2/ngrok.yml)
ngrok_config_path = os.path.expanduser("~/.ngrok2/ngrok.yml")
if os.path.exists(ngrok_config_path):
    print(f"Removing ngrok config file: {ngrok_config_path}")
    os.remove(ngrok_config_path)

# --- End Enhanced Cleanup ---

# Set your ngrok authtoken.
ngrok.set_auth_token("3FXkJDsAj9osZANxjN0KtlK0Qkc_3PSiqjKHNzPu3idjY39Pb")

# Find a free port
streamlit_port = find_free_port()
print(f"Selected Streamlit port: {streamlit_port}")

# Ensure the dynamically found port is truly free before starting Streamlit
kill_process_on_port(streamlit_port);

public_url = None

# Run the Streamlit app in the background using nohup
# Setting STREAMLIT_SERVER_ENABLE_CORS to false helps with embedding in some environments like Colab.
# Using nohup to ensure the process continues running even if the Colab notebook disconnects.
streamlit_process_cmd = [
    "python", # Explicitly use python interpreter
    "-m", "streamlit", # Run streamlit as a module
    "run",
    "app.py",
    "--server.port", str(streamlit_port),
    "--server.address", "127.0.0.1", # Explicitly bind to IPv4 localhost
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false" # Add this to avoid potential CSRF issues in Colab
]
# Execute as a detached process
process = subprocess.Popen(streamlit_process_cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, preexec_fn=os.setsid)

print(f"Streamlit app is starting on port {streamlit_port} at 127.0.0.1...")
time.sleep(5) # Give Streamlit some time to start up

max_retries = 5
for i in range(max_retries):
    try:
        print(f"Attempt {i+1}/{max_retries} to connect ngrok on 127.0.0.1:{streamlit_port}...")
        # Use proto="http" explicitly for Streamlit and specify IPv4 address
        public_url = ngrok.connect(addr=f"127.0.0.1:{streamlit_port}", proto="http", host_header="127.0.0.1")
        break
    except Exception as e:
        print(f"Ngrok connection failed on attempt {i+1}: {e}")
        if i < max_retries - 1:
            print("Retrying in 10 seconds (with additional cleanup)...") # Increased delay
            # Additional cleanup before retry, particularly for server-side issues
            ngrok.kill() # Ensure local ngrok process is killed before next attempt
            if os.path.exists(ngrok_config_path):
                os.remove(ngrok_config_path) # Try removing config again
            time.sleep(10) # Longer sleep
        else:
            print("Max retries reached. Could not establish ngrok tunnel.")
            raise

if public_url:
    print(f"Streamlit App URL: {public_url}")
    print("The Streamlit process will continue to run until this Colab runtime is stopped or restarted.")
else:
    print("Failed to start Streamlit app due to ngrok connection issues.")

Killing all ngrok processes...
Selected Streamlit port: 33783
Streamlit app is starting on port 33783 at 127.0.0.1...
Attempt 1/5 to connect ngrok on 127.0.0.1:33783...
Streamlit App URL: NgrokTunnel: "https://unworldly-padding-mosaic.ngrok-free.dev" -> "http://127.0.0.1:33783"
The Streamlit process will continue to run until this Colab runtime is stopped or restarted.
